# Exploratory Data Analysis

This notebook consumes the **processed tensors** produced by `4_Building_Conditioning_Tensor.ipynb`:

- `data/processed/splits/train.nc`
- `data/processed/splits/val.nc`
- `data/processed/splits/test.nc`
- `data/processed/static_lsm.nc`
- `data/processed/norm_stats.json`

It defines a PyTorch `Dataset` / `DataLoader` pair for the conditional diffusion proof of concept, then performs exploratory data analysis focused on:

1. **Seasonality** of monthly burned fraction.
2. **Spatial heatmaps** of burned-fraction intensity and fire occurrence frequency.
3. **Burned-fraction distribution**, including zero inflation and heavy-tail behaviour.

- Target is monthly burned fraction in `[0, 1]` on a 0.25° Iberian grid.
- Conditioning uses 4 antecedent climate months plus static and calendar channels.
- Splits are year-blocked: train, validation, and test are already stored separately in NetCDF.
- The land-sea mask is used for plotting and summary statistics so ocean cells do not dominate EDA.


> **Update:** this version builds and saves an Iberian-domain mask at `data/processed/static_iberian_mask.nc`, then uses that mask for all EDA summaries and maps. The mask starts from the ERA5 land–sea mask but removes the North African coast that appears inside the rectangular ERA5 bounding box.

## 0. Imports and configuration


In [ ]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
import xarray as xr

import torch
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("default")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


In [ ]:
def find_processed_data_dir() -> Path:
    """Find the processed data directory used by the conditioning-tensor notebook."""
    env = os.environ.get("WILDFIRE_DATA_DIR")
    if env:
        return Path(env).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "processed",
        cwd.parent / "data" / "processed",
        Path("/mnt/data/data/processed"),
    ]
    for candidate in candidates:
        if (candidate / "splits" / "train.nc").exists():
            return candidate.resolve()

    # Fall back to the standard project-relative location. The assertions below
    # will give a clear error if the files are not there yet.
    return (cwd / "data" / "processed").resolve()


DATA_DIR = find_processed_data_dir()
SPLIT_DIR = DATA_DIR / "splits"
SPLIT_FILES = {
    "train": SPLIT_DIR / "train.nc",
    "val": SPLIT_DIR / "val.nc",
    "test": SPLIT_DIR / "test.nc",
}
LSM_PATH = DATA_DIR / "static_lsm.nc"
NORM_STATS_PATH = DATA_DIR / "norm_stats.json"
IBERIAN_MASK_PATH = DATA_DIR / "static_iberian_mask.nc"

LAND_THRESHOLD = 0.5       # continuous ERA5 LSM > 0.5 is treated as land for summaries
BURN_THRESHOLD = 1e-6      # anything above this is treated as a non-zero burned fraction
BATCH_SIZE = 8
NUM_WORKERS = 0            # xarray + NetCDF is safest with 0 workers in notebooks

required_paths = [*SPLIT_FILES.values(), LSM_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing processed tensor files. Expected files under "
        f"{DATA_DIR}. Missing:\n" + "\n".join(missing)
    )

print(f"Using DATA_DIR = {DATA_DIR}")
for name, path in SPLIT_FILES.items():
    print(f"{name:>5}: {path}")
print(f"  lsm: {LSM_PATH}")
print(f" mask: {IBERIAN_MASK_PATH}  # created below if missing")
print(f"stats: {NORM_STATS_PATH if NORM_STATS_PATH.exists() else 'not found'}")


## 1. Inspect the processed tensor files

Each split should contain:

- `conditioning(time, channel, lat, lon)`
- `target(time, lat, lon)`
- a `channel` coordinate naming the conditioning channels


In [ ]:
def describe_split_file(path: Path) -> dict:
    ds = xr.open_dataset(path)
    try:
        out = {
            "path": str(path),
            "sizes": dict(ds.sizes),
            "data_vars": list(ds.data_vars),
            "time_start": str(pd.Timestamp(ds.time.values[0]).date()),
            "time_end": str(pd.Timestamp(ds.time.values[-1]).date()),
            "n_channels": int(ds.sizes.get("channel", -1)),
        }
        if "channel" in ds.coords:
            out["first_channels"] = [str(c) for c in ds.channel.values[:8]]
            out["last_channels"] = [str(c) for c in ds.channel.values[-5:]]
        return out
    finally:
        ds.close()

summary_rows = []
for split_name, path in SPLIT_FILES.items():
    info = describe_split_file(path)
    summary_rows.append({
        "split": split_name,
        "n_time": info["sizes"].get("time"),
        "n_channel": info["n_channels"],
        "n_lat": info["sizes"].get("lat"),
        "n_lon": info["sizes"].get("lon"),
        "start": info["time_start"],
        "end": info["time_end"],
    })

display(pd.DataFrame(summary_rows))

with xr.open_dataset(SPLIT_FILES["train"]) as train_preview:
    channel_names = [str(c) for c in train_preview.channel.values]

print(f"Detected {len(channel_names)} conditioning channels.")
print(channel_names)


In [ ]:
if NORM_STATS_PATH.exists():
    with open(NORM_STATS_PATH, "r") as f:
        norm_stats = json.load(f)
    norm_summary = pd.DataFrame(norm_stats).T
    display(norm_summary)
else:
    norm_stats = {}
    print("No norm_stats.json found. Dataset loading and target EDA do not require it.")


## 2. PyTorch Dataset and DataLoader

The dataset returns a dictionary with:

- `conditioning`: `float32` tensor with shape `[C, H, W]`
- `target`: `float32` tensor with shape `[1, H, W]`
- `land_mask`: `float32` tensor with shape `[H, W]`, repeated per sample for convenience
- `time`, `year`, `month`, and `split` metadata

For diffusion training, `conditioning` is the side information. The noisy target channel is normally created inside the training loop, then concatenated with the conditioning tensor before passing into the U-Net.


In [ ]:
def load_lsm(lsm_path: Path = LSM_PATH) -> xr.DataArray:
    """Load the static land-sea mask as a DataArray sorted by latitude."""
    ds = xr.open_dataset(lsm_path)
    if "lsm" in ds.data_vars:
        da = ds["lsm"]
    else:
        # Defensive fallback in case the variable name changes.
        first_var = list(ds.data_vars)[0]
        da = ds[first_var]
    da = da.squeeze(drop=True)
    if "lat" in da.coords:
        da = da.sortby("lat")
    return da.astype("float32")


def _iberian_southern_boundary(lsm: xr.DataArray) -> xr.DataArray:
    """Piecewise southern boundary used to separate Iberia/Balearics from North Africa.

    This is a pragmatic domain mask for the current proof-of-concept grid. It is not
    a political boundary. The goal is simply to keep Iberian land cells and remove
    North African coastal cells that fall inside the rectangular ERA5 download bbox.
    """
    lat2d, lon2d = xr.broadcast(lsm["lat"], lsm["lon"])
    boundary = xr.full_like(lon2d, 36.0, dtype="float32")

    # Southern Iberia / Strait of Gibraltar region.
    boundary = xr.where((lon2d >= -5.5) & (lon2d < -2.0), 36.25, boundary)
    boundary = xr.where((lon2d >= -2.0) & (lon2d < 0.5), 36.50, boundary)

    # Western Mediterranean: keep eastern Spain and Balearics; drop Algerian coast.
    boundary = xr.where((lon2d >= 0.5) & (lon2d < 1.5), 38.00, boundary)
    boundary = xr.where(lon2d >= 1.5, 38.50, boundary)

    boundary = boundary.rename("southern_boundary_lat")
    boundary.attrs.update({
        "description": "Heuristic southern latitude boundary used to remove North African coastal cells from the rectangular Iberian ERA5 grid.",
        "units": "degrees_north",
    })
    return boundary


def build_iberian_mask(lsm: xr.DataArray, land_threshold: float = LAND_THRESHOLD) -> xr.Dataset:
    """Build the reusable Iberian-domain mask on the processed 0.25° grid.

    Returns a Dataset containing:
    - iberian_mask: uint8, 1 for cells used by EDA/training, 0 otherwise.
    - southern_boundary_lat: the heuristic latitude boundary used for provenance.
    """
    lsm = lsm.sortby("lat") if "lat" in lsm.coords else lsm
    lsm_land = lsm > land_threshold
    lat2d, _ = xr.broadcast(lsm["lat"], lsm["lon"])
    southern_boundary = _iberian_southern_boundary(lsm)

    # Keep ERA5 land cells on the European/Balearic side of the boundary.
    mask = (lsm_land & (lat2d >= southern_boundary)).astype("uint8").rename("iberian_mask")
    mask.attrs.update({
        "description": "Iberian-domain mask. 1 = include in EDA/training loss; 0 = ocean or excluded North African coast.",
        "source_lsm": str(LSM_PATH),
        "land_threshold": float(land_threshold),
        "construction": "ERA5 LSM > land_threshold AND latitude >= piecewise southern Iberia/Balearics boundary.",
        "note": "Heuristic mask for the current 32x54 proof-of-concept grid; replace with a vector country/region mask if a stricter political boundary is required.",
    })

    ds = xr.Dataset({
        "iberian_mask": mask,
        "southern_boundary_lat": southern_boundary.astype("float32"),
    })
    ds.attrs.update({
        "title": "Iberian-domain mask for wildfire forecasting proof of concept",
        "created_by": "5_Exploratory_Data_Analysis_IberianMask.ipynb",
        "purpose": "Remove North African coastal land cells from the rectangular ERA5 grid before EDA/training.",
    })
    return ds


def save_iberian_mask(mask_ds: xr.Dataset, mask_path: Path = IBERIAN_MASK_PATH) -> None:
    """Write the Iberian-domain mask to disk atomically enough for notebook use."""
    mask_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = mask_path.with_name(mask_path.stem + ".tmp" + mask_path.suffix)
    if tmp_path.exists():
        tmp_path.unlink()
    mask_ds.to_netcdf(tmp_path)
    tmp_path.replace(mask_path)


def ensure_iberian_mask(mask_path: Path = IBERIAN_MASK_PATH, lsm_path: Path = LSM_PATH, rebuild: bool = False) -> xr.DataArray:
    """Load the saved Iberian mask, or build and save it if it does not exist."""
    if mask_path.exists() and not rebuild:
        ds = xr.open_dataset(mask_path)
        mask = ds["iberian_mask"].astype("uint8")
        if "lat" in mask.coords:
            mask = mask.sortby("lat")
        return mask

    lsm = load_lsm(lsm_path)
    mask_ds = build_iberian_mask(lsm)
    save_iberian_mask(mask_ds, mask_path)
    return mask_ds["iberian_mask"]


# Ensure the mask exists before constructing DataLoader objects.
_iberian_mask_preview = ensure_iberian_mask(IBERIAN_MASK_PATH, LSM_PATH)
print(f"Iberian mask ready: {IBERIAN_MASK_PATH}")
print("Iberian-domain cells:", int(_iberian_mask_preview.sum().values), "of", _iberian_mask_preview.size)


class WildfireTensorDataset(Dataset):
    """PyTorch Dataset for processed wildfire tensors.

    By default, `sample["land_mask"]` is the saved Iberian-domain mask rather than
    the raw ERA5 LSM threshold. This keeps the model/loss diagnostics aligned with
    the EDA domain and excludes North African coastal cells.
    """

    def __init__(
        self,
        split: str,
        data_dir: Path = DATA_DIR,
        target_unsqueeze: bool = True,
        include_land_mask: bool = True,
        load_into_memory: bool = False,
        fill_nans: bool = False,
    ):
        if split not in {"train", "val", "test"}:
            raise ValueError(f"split must be one of train/val/test, got {split!r}")

        self.split = split
        self.path = Path(data_dir) / "splits" / f"{split}.nc"
        if not self.path.exists():
            raise FileNotFoundError(self.path)

        self.ds = xr.open_dataset(self.path)
        expected = {"conditioning", "target"}
        missing = expected - set(self.ds.data_vars)
        if missing:
            raise KeyError(f"{self.path} is missing variables: {sorted(missing)}")

        # Keep these as DataArrays. They can remain lazy unless load_into_memory=True.
        self.conditioning = self.ds["conditioning"]
        self.target = self.ds["target"]
        if load_into_memory:
            self.conditioning = self.conditioning.load()
            self.target = self.target.load()

        self.times = pd.DatetimeIndex(self.ds["time"].values)
        self.channel_names = [str(c) for c in self.ds["channel"].values]
        self.target_unsqueeze = target_unsqueeze
        self.include_land_mask = include_land_mask
        self.fill_nans = fill_nans

        self.lat = self.ds["lat"].values
        self.lon = self.ds["lon"].values

        if include_land_mask:
            mask_da = ensure_iberian_mask(Path(data_dir) / "static_iberian_mask.nc", Path(data_dir) / "static_lsm.nc")
            mask_da = mask_da.reindex(lat=self.lat, lon=self.lon, method="nearest")
            mask = (mask_da.values > 0).astype("float32")
            self.land_mask = torch.from_numpy(np.ascontiguousarray(mask))
        else:
            self.land_mask = None

    def __len__(self) -> int:
        return int(self.ds.sizes["time"])

    def __getitem__(self, idx: int) -> dict:
        x = np.asarray(self.conditioning.isel(time=idx).values, dtype=np.float32)
        y = np.asarray(self.target.isel(time=idx).values, dtype=np.float32)

        if self.fill_nans:
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
            y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

        x_t = torch.from_numpy(np.ascontiguousarray(x))
        y_t = torch.from_numpy(np.ascontiguousarray(y))
        if self.target_unsqueeze:
            y_t = y_t.unsqueeze(0)

        timestamp = pd.Timestamp(self.times[idx])
        sample = {
            "conditioning": x_t,
            "target": y_t,
            "time": timestamp.strftime("%Y-%m-%d"),
            "year": int(timestamp.year),
            "month": int(timestamp.month),
            "split": self.split,
        }
        if self.include_land_mask:
            sample["land_mask"] = self.land_mask
        return sample

    def close(self) -> None:
        self.ds.close()


In [ ]:
train_ds = WildfireTensorDataset("train", fill_nans=False)
val_ds = WildfireTensorDataset("val", fill_nans=False)
test_ds = WildfireTensorDataset("test", fill_nans=False)

for ds_name, ds_obj in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    sample = ds_obj[0]
    print(
        f"{ds_name:>5}: n={len(ds_obj):3d}, "
        f"x={tuple(sample['conditioning'].shape)}, "
        f"y={tuple(sample['target'].shape)}, "
        f"first_time={sample['time']}"
    )

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

batch = next(iter(train_loader))
print("\nExample batch:")
print("conditioning", tuple(batch["conditioning"].shape), batch["conditioning"].dtype)
print("target      ", tuple(batch["target"].shape), batch["target"].dtype)
print("land_mask   ", tuple(batch["land_mask"].shape), batch["land_mask"].dtype)
print("time        ", batch["time"][:3])

print("\nNaN checks on this batch:")
print("conditioning NaNs:", torch.isnan(batch["conditioning"]).sum().item())
print("target NaNs:      ", torch.isnan(batch["target"]).sum().item())


### Optional: one training-loop-shaped batch

This is not a diffusion implementation; it is just a shape check showing where the noisy target would be concatenated with the conditioning tensor during training.


In [ ]:
# Example only: add Gaussian noise to the target and concatenate with conditioning.
# A real diffusion trainer would sample a timestep and use the corresponding noise schedule.
x_cond = batch["conditioning"]          # [B, C, H, W]
y0 = batch["target"]                   # [B, 1, H, W]
noise = torch.randn_like(y0)
noisy_target = y0 + 0.1 * noise
unet_input = torch.cat([noisy_target, x_cond], dim=1)

print("U-Net input shape if target noise channel is concatenated:", tuple(unet_input.shape))
print("Expected channels = 1 noisy target + conditioning channels =", 1 + len(train_ds.channel_names))


## 3. EDA helper functions

The target is a burned **fraction** per grid cell. For regional summaries, the notebook uses approximate area weights:

\[
weight(lat, lon) \propto \cos(latitude) \times LSM(lat, lon)
\]

This is sufficient for EDA on a regular lat/lon grid and avoids giving ocean cells or tiny coastal fractions the same weight as full land cells.


In [ ]:
lsm = load_lsm(LSM_PATH)
lsm_land_mask = lsm > LAND_THRESHOLD
iberian_mask_da = ensure_iberian_mask(IBERIAN_MASK_PATH, LSM_PATH)
iberian_mask_da = iberian_mask_da.reindex(lat=lsm.lat, lon=lsm.lon, method="nearest")
land_mask = iberian_mask_da > 0
excluded_lsm_land = lsm_land_mask & ~land_mask

# Approximate cell-area weight for regular lon/lat grid, scaled by continuous land fraction.
# The .where(land_mask) term means every downstream summary is Iberian-domain-only.
lat_weights = xr.DataArray(
    np.cos(np.deg2rad(lsm["lat"].values)).astype("float64"),
    dims=("lat",),
    coords={"lat": lsm["lat"]},
)
area_weights = (lsm.clip(0, 1).astype("float64") * lat_weights).where(land_mask)
area_weights = area_weights / area_weights.sum()

print("LSM shape:", tuple(lsm.shape))
print("Raw LSM land cells above threshold:", int(lsm_land_mask.sum().values), "of", lsm_land_mask.size)
print("Iberian-domain cells used for EDA/training mask:", int(land_mask.sum().values), "of", land_mask.size)
print("Excluded LSM-land cells, mostly North African coast:", int(excluded_lsm_land.sum().values))
print("Saved Iberian mask:", IBERIAN_MASK_PATH)
print("Area weights sum:", float(area_weights.sum().values))

# QA plot: positive values are retained Iberian cells; negative values are excluded LSM-land cells.
mask_qa = xr.where(land_mask, 1, xr.where(excluded_lsm_land, -1, np.nan)).rename("mask_qa")
fig, ax = plt.subplots(figsize=(8.5, 5))
mask_qa.plot(ax=ax, cbar_kwargs={"label": "1 = retained Iberian land, -1 = excluded LSM land"})
ax.set_title("Iberian-domain mask QA")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


In [ ]:
def open_target_split(split_name: str) -> xr.DataArray:
    ds = xr.open_dataset(SPLIT_FILES[split_name])
    da = ds["target"].astype("float32")
    da = da.assign_coords(split=("time", np.repeat(split_name, da.sizes["time"])))
    if "lat" in da.coords:
        da = da.sortby("lat")
    return da


target_all = xr.concat(
    [open_target_split(split_name) for split_name in ["train", "val", "test"]],
    dim="time",
).sortby("time")

target_land = target_all.where(land_mask)

print("Combined target:", target_all.sizes)
print("Time range:", str(pd.Timestamp(target_all.time.values[0]).date()), "→", str(pd.Timestamp(target_all.time.values[-1]).date()))
print("Splits:")
print(pd.Series(target_all["split"].values).value_counts().to_string())


In [ ]:
def weighted_spatial_mean(da: xr.DataArray, weights: xr.DataArray = area_weights) -> xr.DataArray:
    """Weighted mean over lat/lon, preserving non-spatial dimensions such as time."""
    return (da.fillna(0) * weights).sum(dim=("lat", "lon"))


def weighted_active_fraction(da: xr.DataArray, threshold: float = BURN_THRESHOLD) -> xr.DataArray:
    """Weighted fraction of land area with burned fraction above threshold."""
    active = (da > threshold).astype("float32").where(land_mask)
    return (active.fillna(0) * area_weights).sum(dim=("lat", "lon"))


def make_monthly_timeseries(target_da: xr.DataArray) -> pd.DataFrame:
    """Create one row per target month with regional burned-fraction diagnostics."""
    mean_burned = weighted_spatial_mean(target_da.where(land_mask))
    active_frac = weighted_active_fraction(target_da)
    max_cell = target_da.where(land_mask).max(dim=("lat", "lon"), skipna=True)
    q95_cell = target_da.where(land_mask).quantile(0.95, dim=("lat", "lon"), skipna=True)
    q99_cell = target_da.where(land_mask).quantile(0.99, dim=("lat", "lon"), skipna=True)

    index = pd.DatetimeIndex(target_da.time.values)
    df = pd.DataFrame({
        "time": index,
        "split": target_da["split"].values,
        "year": index.year,
        "month": index.month,
        "mean_burned_fraction": mean_burned.values,
        "active_land_fraction": active_frac.values,
        "p95_cell_burned_fraction": q95_cell.values,
        "p99_cell_burned_fraction": q99_cell.values,
        "max_cell_burned_fraction": max_cell.values,
    })
    return df.sort_values("time").reset_index(drop=True)


ts = make_monthly_timeseries(target_all)
display(ts.head())
display(ts.groupby("split")[["mean_burned_fraction", "active_land_fraction", "max_cell_burned_fraction"]].describe())


## 4. Seasonality

The first question is whether the target shows the expected Iberian fire-season structure: low winter activity, rising spring/summer activity, and a strong summer peak.


In [ ]:
seasonal = (
    ts.groupby(["split", "month"], as_index=False)
      [["mean_burned_fraction", "active_land_fraction", "p99_cell_burned_fraction"]]
      .mean()
)

fig, ax = plt.subplots(figsize=(9, 4.8))
for split_name, group in seasonal.groupby("split"):
    group = group.sort_values("month")
    ax.plot(group["month"], group["mean_burned_fraction"], marker="o", label=split_name)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Target month")
ax.set_ylabel("Area-weighted mean burned fraction")
ax.set_title("Monthly seasonality of regional burned fraction")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for split_name, group in seasonal.groupby("split"):
    group = group.sort_values("month")
    ax.plot(group["month"], group["active_land_fraction"], marker="o", label=split_name)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Target month")
ax.set_ylabel(f"Area fraction with burned_fraction > {BURN_THRESHOLD:g}")
ax.set_title("Seasonality of active burned land fraction")
ax.legend()
plt.show()


In [ ]:
monthly_summary = (
    ts.groupby("month")
      .agg(
          mean_burned_fraction=("mean_burned_fraction", "mean"),
          median_burned_fraction=("mean_burned_fraction", "median"),
          active_land_fraction=("active_land_fraction", "mean"),
          max_cell_burned_fraction=("max_cell_burned_fraction", "max"),
      )
      .reset_index()
)
display(monthly_summary)


In [ ]:
ym = ts.pivot_table(index="year", columns="month", values="mean_burned_fraction", aggfunc="mean")
fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(ym.values, aspect="auto")
fig.colorbar(im, ax=ax, label="Area-weighted mean burned fraction")
ax.set_title("Year × month heatmap of regional burned fraction")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
ax.set_xticks(np.arange(12))
ax.set_xticklabels(range(1, 13))
ax.set_yticks(np.arange(len(ym.index)))
ax.set_yticklabels(ym.index)
plt.show()


In [ ]:
top_months = ts.sort_values("mean_burned_fraction", ascending=False).head(15)
display(top_months[[
    "time", "split", "mean_burned_fraction", "active_land_fraction",
    "p99_cell_burned_fraction", "max_cell_burned_fraction"
]])


## 5. Spatial heatmaps

These maps show where burned fraction is concentrated and how frequently grid cells burn. They are plotted in the raw lat/lon grid without cartographic projection to keep dependencies minimal.


In [ ]:
def plot_map(da: xr.DataArray, title: str, label: str = "burned fraction", robust: bool = True):
    da = da.where(land_mask)
    fig, ax = plt.subplots(figsize=(8.5, 5))
    da.plot(ax=ax, robust=robust, cbar_kwargs={"label": label})
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.show()


mean_map = target_land.mean("time", skipna=True)
active_frequency_map = (target_all > BURN_THRESHOLD).astype("float32").where(land_mask).mean("time", skipna=True)
max_map = target_land.max("time", skipna=True)

plot_map(mean_map, "Mean monthly burned fraction, Iberian mask, all splits", "mean burned fraction")
plot_map(active_frequency_map, f"Frequency of burned cells, Iberian mask, threshold > {BURN_THRESHOLD:g}", "fraction of months active", robust=False)
plot_map(max_map, "Maximum observed cell burned fraction, Iberian mask, all splits", "max burned fraction")


In [ ]:
for split_name in ["train", "val", "test"]:
    split_map = target_all.where(target_all["split"] == split_name, drop=True).where(land_mask).mean("time", skipna=True)
    plot_map(split_map, f"Mean monthly burned fraction — {split_name}", "mean burned fraction")


In [ ]:
# Seasonal spatial means. xarray's time.season uses DJF/MAM/JJA/SON.
season_order = ["DJF", "MAM", "JJA", "SON"]
seasonal_maps = target_land.groupby("time.season").mean("time", skipna=True)

for season in season_order:
    if season in seasonal_maps["season"].values:
        plot_map(
            seasonal_maps.sel(season=season),
            f"Mean monthly burned fraction — {season}",
            "mean burned fraction",
        )


In [ ]:
# Case-study months. These are included only if present in the processed tensors.
case_study_months = ["2017-06-01", "2017-08-01", "2017-10-01", "2022-07-01", "2022-08-01"]
available_times = set(pd.DatetimeIndex(target_all.time.values))

for date_str in case_study_months:
    date = pd.Timestamp(date_str)
    if date in available_times:
        da = target_all.sel(time=date).where(land_mask)
        split_name = str(target_all.sel(time=date)["split"].values)
        plot_map(da, f"Burned fraction target — {date:%Y-%m} ({split_name})", "burned fraction", robust=False)
    else:
        print(f"Skipping {date_str}: not present in target_all")


## 6. Burned-fraction distribution

Wildfire burned fraction is usually **zero-inflated** and **heavy-tailed**: most land cell-months have no burning, while a small number of cells/months carry the fire signal. This matters for diffusion training, baselines, threshold metrics, and visual diagnostics.


In [ ]:
def land_values_for_split(split_name: str) -> np.ndarray:
    da = target_all.where(target_all["split"] == split_name, drop=True).where(land_mask)
    values = np.asarray(da.values).ravel()
    values = values[np.isfinite(values)]
    return values.astype("float64")


def summarize_distribution(values: np.ndarray, split_name: str) -> dict:
    positives = values[values > BURN_THRESHOLD]
    row = {
        "split": split_name,
        "n_land_cell_months": int(values.size),
        "zero_or_tiny_%": 100 * float(np.mean(values <= BURN_THRESHOLD)),
        "positive_%": 100 * float(np.mean(values > BURN_THRESHOLD)),
        "mean_all": float(np.mean(values)) if values.size else np.nan,
        "median_all": float(np.median(values)) if values.size else np.nan,
        "max_all": float(np.max(values)) if values.size else np.nan,
    }
    if positives.size:
        row.update({
            "n_positive": int(positives.size),
            "mean_positive": float(np.mean(positives)),
            "median_positive": float(np.median(positives)),
            "p90_positive": float(np.quantile(positives, 0.90)),
            "p95_positive": float(np.quantile(positives, 0.95)),
            "p99_positive": float(np.quantile(positives, 0.99)),
        })
    else:
        row.update({
            "n_positive": 0,
            "mean_positive": np.nan,
            "median_positive": np.nan,
            "p90_positive": np.nan,
            "p95_positive": np.nan,
            "p99_positive": np.nan,
        })
    return row

values_by_split = {split_name: land_values_for_split(split_name) for split_name in ["train", "val", "test"]}
dist_summary = pd.DataFrame([
    summarize_distribution(values, split_name)
    for split_name, values in values_by_split.items()
])
display(dist_summary)


In [ ]:
zero_rates = dist_summary.set_index("split")["zero_or_tiny_%"]
fig, ax = plt.subplots(figsize=(7, 4.5))
zero_rates.plot(kind="bar", ax=ax)
ax.set_ylabel(f"% land cell-months <= {BURN_THRESHOLD:g}")
ax.set_title("Zero inflation by split")
ax.set_xlabel("Split")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Histogram of strictly positive burned fractions on log-log axes.
positive_arrays = {
    split_name: values[values > BURN_THRESHOLD]
    for split_name, values in values_by_split.items()
}
all_positive_parts = [v for v in positive_arrays.values() if v.size]
all_positive = np.concatenate(all_positive_parts) if all_positive_parts else np.array([])

if all_positive.size:
    lower = max(BURN_THRESHOLD, float(all_positive.min()))
    upper = float(all_positive.max())
    bins = np.geomspace(lower, upper, 60) if upper > lower else 30

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for split_name, positives in positive_arrays.items():
        if positives.size:
            ax.hist(positives, bins=bins, density=True, histtype="step", linewidth=1.5, label=split_name)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Positive burned fraction")
    ax.set_ylabel("Density")
    ax.set_title("Distribution of positive burned fractions")
    ax.legend()
    plt.show()
else:
    print("No positive burned-fraction values found above threshold.")


In [ ]:
# Empirical CDF of positive values.
fig, ax = plt.subplots(figsize=(8, 4.8))
for split_name, positives in positive_arrays.items():
    if positives.size:
        x = np.sort(positives)
        y = np.arange(1, len(x) + 1) / len(x)
        ax.plot(x, y, label=split_name)
ax.set_xscale("log")
ax.set_xlabel("Positive burned fraction")
ax.set_ylabel("Empirical CDF")
ax.set_title("ECDF of positive burned fractions")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ts.boxplot(column="mean_burned_fraction", by="month", ax=ax)
ax.set_title("Monthly distribution of regional mean burned fraction")
ax.set_xlabel("Target month")
ax.set_ylabel("Area-weighted mean burned fraction")
plt.suptitle("")
plt.show()


## 7. Batch-level diagnostics for model development

These summaries run through a few `DataLoader` batches and check ranges. Use this before training to catch shape changes, NaNs, or unexpected target values.


In [ ]:
def finite_min(t: torch.Tensor) -> float:
    finite = t[torch.isfinite(t)]
    return float(finite.min()) if finite.numel() else np.nan


def finite_max(t: torch.Tensor) -> float:
    finite = t[torch.isfinite(t)]
    return float(finite.max()) if finite.numel() else np.nan


def finite_mean(t: torch.Tensor) -> float:
    finite = t[torch.isfinite(t)]
    return float(finite.mean()) if finite.numel() else np.nan


def inspect_loader(loader: DataLoader, n_batches: int = 5) -> pd.DataFrame:
    rows = []
    for batch_idx, batch in enumerate(loader):
        if batch_idx >= n_batches:
            break
        x = batch["conditioning"]
        y = batch["target"]
        mask = batch["land_mask"].unsqueeze(1).bool()
        y_land = y.masked_select(mask)
        rows.append({
            "batch": batch_idx,
            "batch_size": int(x.shape[0]),
            "x_min": finite_min(x),
            "x_max": finite_max(x),
            "x_nan_count": int(torch.isnan(x).sum()),
            "y_min_land": finite_min(y_land),
            "y_max_land": finite_max(y_land),
            "y_mean_land": finite_mean(y_land),
            "y_nan_count": int(torch.isnan(y).sum()),
        })
    return pd.DataFrame(rows)

loader_diagnostics = inspect_loader(train_loader, n_batches=5)
display(loader_diagnostics)


## 8. EDA takeaways

### 8.1 Which months dominate the seasonal cycle?

The seasonal cycle is strongly summer-dominated. August is the main peak in regional mean burned fraction, with July, September, and October also carrying most of the fire signal. June marks the beginning of the main season but is weaker than the July–October window.

For modelling and evaluation, report at least an all-year score and a fire-season score. A practical fire-season grouping for this project is **July–October**, with August tracked separately because it is the dominant month.

### 8.2 Are the 2017 and 2022 case-study months among the largest regional burned-fraction months?

Yes. The first executed EDA run showed **2017-10** and **2022-07** among the largest regional burned-fraction months, with **2017-08** and **2022-08** also important. 

### 8.3 Which spatial regions are persistent hotspots versus rare extreme cells?

The persistent hotspots are concentrated in western and northwestern Iberia, especially Portugal and the Galicia / northwest-Iberia region. These areas appear in both the mean burned-fraction map and the active-frequency map, meaning they are not just one-off extremes.

### 8.4 How zero-inflated is the target, and how different are train/validation/test tail quantiles?

The target is extremely zero-inflated: the first run showed roughly **97–98%** of Iberian/land cell-months at zero or near-zero burned fraction. This is expected for monthly burned fraction at 0.25° resolution.

The positive tail is much more informative than the all-cell distribution. In the first run, the test split had the heaviest positive tail, while validation was much lighter-tailed. That means validation is useful for general sanity checks, but it may be weak for selecting models based on extreme-fire skill.

### 8.5 Does the distribution suggest adding loss weighting, tail-focused validation diagnostics, or transformed target diagnostics for baselines?

Yes. Because the target is so zero-inflated, global RMSE/MAE can look good while the model fails the rare high-burned-fraction cells that matter most.

- report metrics for all cells, positive cells only, and the top 1% / top 5% burned-fraction tail;
- report fire-season metrics separately, especially July–October;
- report case-study diagnostics for 2017-10, 2022-07, and 2022-08;
- add threshold/event diagnostics such as `burned_fraction > 0`, `> 0.01`, and `> 0.05`;
- for deterministic baselines, test positive-cell weighting or high-fire-month oversampling;
- inspect transformed-target diagnostics for baselines, e.g. `sqrt(y)` or `log1p(alpha*y)`, while keeping raw burned fraction as the primary interpretable target.

For the diffusion model itself, the most important practical rule is: **fill NaNs before passing tensors to the U-Net, then apply the saved Iberian mask in the loss and metrics.**


In [ ]:
# Compact, data-backed tables for the takeaways after rerunning with the Iberian mask.
# These are intentionally recomputed from the current masked EDA objects.

seasonal_rank = monthly_summary.sort_values("mean_burned_fraction", ascending=False).head(6)
seasonal_rank = seasonal_rank[[
    "month", "mean_burned_fraction", "median_burned_fraction",
    "active_land_fraction", "max_cell_burned_fraction"
]]

ranked_ts = ts.sort_values("mean_burned_fraction", ascending=False).reset_index(drop=True)
ranked_ts["rank"] = np.arange(1, len(ranked_ts) + 1)
case_dates = pd.to_datetime(["2017-06-01", "2017-08-01", "2017-10-01", "2022-07-01", "2022-08-01"])
case_rank = ranked_ts[ranked_ts["time"].isin(case_dates)][[
    "rank", "time", "split", "mean_burned_fraction", "active_land_fraction",
    "p99_cell_burned_fraction", "max_cell_burned_fraction"
]].sort_values("rank")

compact_dist = dist_summary[[
    "split", "n_land_cell_months", "zero_or_tiny_%", "positive_%",
    "max_all", "p95_positive", "p99_positive"
]]

display(Markdown("### Data-backed takeaway tables after applying the Iberian mask"))
display(Markdown("**Dominant target months by regional mean burned fraction**"))
display(seasonal_rank)
display(Markdown("**Ranks of selected 2017/2022 case-study months**"))
display(case_rank)
display(Markdown("**Zero inflation and positive-tail quantiles by split**"))
display(compact_dist)
